In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
import os

# CONFIGURATION & EXPLICIT MAPPINGS
INPUT_CSV = "ML_Training_Dataset_Cleaned.xlsx" 
OUTPUT_CSV = "Master_Processed_Features.xlsx"

NOMINAL_COLS = ['Image_Context']

ORDINAL_COLS = [
    'Housing_Category', 'Ext_Stories', 'Ext_Roof_Type', 
    'Ext_Wall_Type', 'Ext_Structural_Condition',
    'Int_Floor_Material', 'Int_Wall_Finish'
]

REGRESSION_COLS = ['Overall_Structural_Score']

ORDINAL_MAPS = {
    'Housing_Category': {'Kutcha': 0, 'Semi-Pucca': 1, 'Pucca': 2, 'Premium': 3},
    'Ext_Stories': {'One Storied': 0, 'Two Storied': 1, 'Three Storied': 2, 'Four Storied': 3, 'Five Storied': 4, 'Six Storied': 5, 'Seven Storied': 6, 'Eight Storied': 7, 'Nine Storied': 8, 'Ten Storied': 9},
    'Ext_Roof_Type': {'Thatch/Tarpaulin': 0, 'Corrugated Tin/Metal': 1, 'Khaprail': 2, 'Asbestos': 3, 'Concrete': 4},
    'Ext_Wall_Type': {'Mud/Makeshift': 0, 'Exposed Brick': 1, 'Finished Concrete/Plaster': 2},
    'Ext_Structural_Condition': {'Dilapidated': 0, 'Poor': 1, 'Average': 2, 'Excellent': 3},
    'Int_Floor_Material': {'Mud/Earth': 0, 'Cement/Concrete': 1, 'Tiles/Marble': 2},
    'Int_Wall_Finish': {'Mud': 0, 'Bare_Brick': 1, 'Raw_Plaster': 2, 'Painted': 3, 'Tiles': 4}
}

# LOADING AND CLEANING
print(f"Loading cleaned dataset: {INPUT_CSV}...")
df = pd.read_excel(INPUT_CSV)

print(f"Processing {len(df)} valid rows...")

# Cleaning up unnecessary columns left over from the extraction phase
columns_to_drop = ['Processing_Status']
for col in columns_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"Dropped '{col}' column to keep features clean.")

# Standardizing text for missing values
ALL_CATS = NOMINAL_COLS + ORDINAL_COLS
df[ALL_CATS] = df[ALL_CATS].fillna("N/A")

# Missing scores become -100.0 so we can mask them in PyTorch
df[REGRESSION_COLS] = df[REGRESSION_COLS].fillna(-100.0) 

# ADVANCED ENCODING (-100 MASKING)
print("Encoding features with -100 masking logic...")
encoding_maps = {} 

# A. NOMINAL ENCODING (With -100 override)
for col in NOMINAL_COLS:
    le = LabelEncoder()
    encoded_col_name = f"{col}_Encoded"
    
    valid_mask = ~df[col].isin(['N/A', 'Unknown', 'nan', ''])
    le.fit(df.loc[valid_mask, col].astype(str))
    
    df[encoded_col_name] = -100
    df.loc[valid_mask, encoded_col_name] = le.transform(df.loc[valid_mask, col].astype(str))
    encoding_maps[col] = dict(zip(le.transform(le.classes_), le.classes_))

# B. ORDINAL ENCODING (Applying strict hierarchies)
for col in ORDINAL_COLS:
    encoded_col_name = f"{col}_Encoded"
    mapping = ORDINAL_MAPS.get(col, {})
    df[encoded_col_name] = df[col].map(mapping).fillna(-100).astype(int)

# MULTI-LABEL BINARIZING (Assets)
print("Expanding visible assets into individual columns...")

df['Asset_List'] = df['Visible_Assets_For_Income'].apply(
    lambda x: [item.strip() for item in str(x).split(',')] if str(x).strip() not in ["None", "", "nan", "[]"] else []
)

mlb = MultiLabelBinarizer()
encoded_assets = mlb.fit_transform(df['Asset_List'])

asset_column_names = [f"Asset_{c.replace(' ', '_')}" for c in mlb.classes_]
df_assets = pd.DataFrame(encoded_assets, columns=asset_column_names, index=df.index)

df = pd.concat([df, df_assets], axis=1)
df = df.drop(columns=['Asset_List'])

# LOGICAL OVERRIDES (The Safety Filter)
print("Applying logical overrides based on Image Context...")

# If it's an Exterior image, force all Interior features to -100
exterior_mask = df['Image_Context'].str.contains('Exterior', na=False)
int_cols = ['Int_Floor_Material_Encoded', 'Int_Wall_Finish_Encoded']
df.loc[exterior_mask, int_cols] = -100

# If it's an Interior image, force all Exterior features to -100
interior_mask = df['Image_Context'].str.contains('Interior', na=False)
ext_cols = [
    'Ext_Stories_Encoded', 'Ext_Roof_Type_Encoded', 
    'Ext_Wall_Type_Encoded'
]
df.loc[interior_mask, ext_cols] = -100

print(f"Forced interior masking on {exterior_mask.sum()} exterior images.")
print(f"Forced exterior masking on {interior_mask.sum()} interior images.")


# SAVING MASTER FILE
print(f"\nSaving processed master file to: {OUTPUT_CSV}")
df.to_excel(OUTPUT_CSV, index=False)

print("\n--- Preprocessing Complete! ---")

Loading cleaned dataset: ML_Training_Dataset_Cleaned.xlsx...
Processing 3676 valid rows...
Dropped 'Processing_Status' column to keep features clean.
Encoding features with -100 masking logic...
Expanding visible assets into individual columns...
Applying logical overrides based on Image Context...
Forced interior masking on 2477 exterior images.
Forced exterior masking on 1199 interior images.

Saving processed master file to: Master_Processed_Features.xlsx

--- Preprocessing Complete! ---


In [2]:
import os
from PIL import Image, ImageOps

# Configuration
INPUT_DIR = "./IMAGES/"
OUTPUT_DIR = "./IMAGES_PADDED/"
TARGET_SIZE = (600, 600)
PADDING_COLOR = (0, 0, 0) # Black padding

# Creating the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"Scanning directory: {INPUT_DIR}")
valid_extensions = ('.jpg', '.jpeg', '.png')
all_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_extensions)]

print(f"Found {len(all_files)} images. Starting letterboxing process...\n")

for index, filename in enumerate(all_files):
    input_path = os.path.join(INPUT_DIR, filename)
    output_path = os.path.join(OUTPUT_DIR, filename)
    
    try:
        # Loading the original image
        img = Image.open(input_path).convert('RGB')
        
        # Applying the letterbox padding
        padded_img = ImageOps.pad(img, TARGET_SIZE, color=PADDING_COLOR)
        
        # Saving to the new folder
        padded_img.save(output_path)
        
        # Reduced print frequency for a cleaner console
        if (index + 1) % 50 == 0:
            print(f"[{index + 1}/{len(all_files)}] Processed...")
            
    except Exception as e:
        print(f"Error processing {filename}: {e}")

print("\n--- Letterboxing Complete! ---")
print(f"All ML-ready images are saved in: {OUTPUT_DIR}")

Scanning directory: ./IMAGES/
Found 3676 images. Starting letterboxing process...

[50/3676] Processed...
[100/3676] Processed...
[150/3676] Processed...
[200/3676] Processed...
[250/3676] Processed...
[300/3676] Processed...
[350/3676] Processed...
[400/3676] Processed...
[450/3676] Processed...
[500/3676] Processed...
[550/3676] Processed...
[600/3676] Processed...
[650/3676] Processed...
[700/3676] Processed...
[750/3676] Processed...
[800/3676] Processed...
[850/3676] Processed...
[900/3676] Processed...
[950/3676] Processed...
[1000/3676] Processed...
[1050/3676] Processed...
[1100/3676] Processed...
[1150/3676] Processed...
[1200/3676] Processed...
[1250/3676] Processed...
[1300/3676] Processed...
[1350/3676] Processed...
[1400/3676] Processed...
[1450/3676] Processed...
[1500/3676] Processed...
[1550/3676] Processed...
[1600/3676] Processed...
[1650/3676] Processed...
[1700/3676] Processed...
[1750/3676] Processed...
[1800/3676] Processed...
[1850/3676] Processed...
[1900/3676] 